# 获取并解压绿色包

这一节做一件事：把 HCCL-VM 的运行环境搬到 cannlab 环境里。

做完之后，你就有了一个能直接跑的仿真器，**不需要安装 CANN，也不需要编译任何代码**。

## 本节学习大纲

- 为什么可以直接用绿色包，不用装 CANN
- 确认运行环境：架构与内存够不够
- 下载绿色包
- 校验文件完整性
- 解压并认识包里的目录
- 生成环境脚本 env.sh
- 遇到问题先看这里

## 1. 为什么可以直接用绿色包

安装 CANN 会往机器上放五类东西，其中只有两类是"运行时"真正要用的：

| 内容 | 什么时候用 | 这一课要不要 |
| :--- | :--- | :--- |
| 头文件 `include/hccl/*.h` | 编译时 | 不要，我们不编译 |
| 编译器、`tools/hcc`、Python 包 | 编译时 | 不要 |
| 驱动 driver、firmware | 有真卡时 | 不要，HCCL-VM 是模拟器 |
| 运行库 `lib64/*.so` | 运行时 | **要** |
| 算子与配置 `opp/.../aicpu/` | 运行时 | **要** |

后两类是纯文件，压缩后只有 82 MB，已经提前打好包放在仓库里了。

包里的 `run_hccl_vm.sh` 会替你做原来 `set_env.sh` 做的事：

```bash
export LD_LIBRARY_PATH="$HERE/lib64:..."   # 动态库去哪找
export ASCEND_HOME_PATH="$HERE"            # 配置与算子去哪找
```

所以"安装 CANN"这件事，在这里变成了"解压一个文件夹"。

## 2. 确认运行环境

在 cannlab 环境里打开课件后，先确认三件事：

| 项 | 要求 | 说明 |
| :--- | :--- | :--- |
| 计算类型 | CPU | 不需要 NPU，模拟器跑在 CPU 上 |
| 架构 | **aarch64** | 绿色包只适用于 aarch64，`uname -m` 应输出 `aarch64` |
| 内存 | 建议 16 GB 以上 | 低于 8 GB 时仿真可能申请不到共享内存 |

**CANN 不用管。** cannlab 环境自带 CANN 9.0.0，但我们的包**不依赖它**：
包内自带一整套运行库，二进制里写了 RPATH，永远优先用包内的库。
所以机器上装的是 9.0.0、9.2.0，还是根本没装，都不影响运行。

In [ ]:
%%bash
uname -m
free -g | head -2

期望输出里出现 `aarch64` 和总内存 `16`（单位 GB）。如果架构是 `x86_64`，说明实例选错了——
绿色包只适用于 aarch64（鲲鹏）实例。

## 3. 下载绿色包

绿色包挂在仓库的 Release 里，用 `wget` 直接下载到你的家目录。

> 如果你的实例上已经有 `~/hccl_vm_pack.tar.gz`（比如你手动上传过），下面的命令会自动跳过下载。

In [ ]:
%%bash
PACK_URL="https://gitcode.com/weixin_45130366/cann-learning-hub_2612/releases/download/v2/hccl_vm_pack.tar.gz"

cd ~
if [ -f hccl_vm_pack.tar.gz ]; then
    echo "hccl_vm_pack.tar.gz already exists, skip download"
else
    wget -q --show-progress "$PACK_URL" -O hccl_vm_pack.tar.gz
fi
ls -lh ~/hccl_vm_pack.tar.gz

文件大小应该是 **82 MB** 左右。如果只有几 KB 或几百字节，说明下载到的是一个错误页面，
把地址复制到浏览器里试一下，确认能下到文件。

## 4. 校验文件完整性

下载过程中断过、或者代理缓存出错，文件可能"看起来下完了、其实是坏的"。
`sha256` 是一串文件指纹，和官方值一致才说明文件完整。

下面这条命令会把本地算出的指纹和期望值比对，输出 `OK` 才算通过。

In [ ]:
%%bash
cd ~
echo "6285999cf3cc0587065ae5f4ca4e9c25ea08c427be48650649c56f3aa472a930  hccl_vm_pack.tar.gz" | sha256sum -c

看到 `hccl_vm_pack.tar.gz: OK` 就继续；如果显示 `FAILED`，删掉文件重新下载一次。

## 5. 解压并认识包里的目录

解压后会出现 `hccl_vm_pack/` 目录，里面八个东西各管一件事。

In [ ]:
%%bash
cd ~
tar xzf hccl_vm_pack.tar.gz
ls hccl_vm_pack

| 目录 / 文件 | 作用 |
| :--- | :--- |
| `hccl_vm_install/` | 模拟器本体：`bin/hccl-vm`（主程序）、`bin/device`、`plugin/`、`data/`、`config/` |
| `hccl_test/` | 算例二进制，例如 `bin/reduce_scatter_test` |
| `lib64/` | 运行库，36 个（含 `libruntime_v100.so`、`hccl_aiv_op_910_95.o`） |
| `opp/` | 算子配置与内核包，`ccl_kernel.json` 在这里 |
| `conf/` | 错误码表 `error_manager/error_code.json` |
| `run_hccl_vm.sh` | 启动器：设好环境变量后调用 `bin/hccl-vm` |
| `run_test.sh` | 一键跑测试的示例脚本 |
| `README.md` | 包的简短说明 |

> `lib64` 这个名字不能改成别的。HCCL 内部是按 `$ASCEND_HOME_PATH/lib64/...` 这个固定路径去找库的，
> 改成 `libs` 之类会导致程序读不到任何库。

## 6. 生成环境脚本 env.sh

Notebook 里每个代码单元都是**独立的一次 shell**，上一个单元 `export` 的变量不会留到下一个单元。

所以我们在包目录下生成一个 `env.sh`，之后每个单元开头都 `source` 它一次，路径变量就都齐了。

In [ ]:
%%bash
cat > ~/hccl_vm_pack/env.sh <<'EOF'
export PACK="$HOME/hccl_vm_pack"
export VM_INSTALL="$PACK/hccl_vm_install"
export VM="$VM_INSTALL/bin/hccl-vm"
export HCCL_VM_INSTALL_DIR="$VM_INSTALL"
export TEST_BIN="$PACK/hccl_test/bin"
export CLUSTER_YAML="ascend950_cluster_32_server_normal"
export RECORD_ROOT="$VM_INSTALL/records"
export HCCL_VM_RESULT_DIR="${HCCL_VM_RESULT_DIR:-$RECORD_ROOT/last}"
EOF
mkdir -p "$HOME/hccl_vm_pack/hccl_vm_install/records"
source ~/hccl_vm_pack/env.sh
echo "PACK        = $PACK"
echo "VM          = $VM"
echo "TEST_BIN    = $TEST_BIN"
echo "CLUSTER_YAML= $CLUSTER_YAML"

四个变量都打印出路径，说明环境脚本可用。后面每一节的第一行都会是 `source ~/hccl_vm_pack/env.sh`。

## 7. 遇到问题先看这里

| 现象 | 原因 | 处理 |
| :--- | :--- | :--- |
| `uname -m` 是 `x86_64` | 实例选了 x86 规格 | 重建实例，选 CPU 规格（鲲鹏 aarch64） |
| 下载只有几 KB | 地址失效或需要登录 | 在浏览器里打开地址确认能下 |
| `sha256sum -c` 显示 FAILED | 文件没下完整 | 删除后重新下载 |
| `tar` 报 `unexpected EOF` | 压缩包被截断 | 同上，重新下载并再次校验 |
| `ls hccl_vm_pack` 少目录 | 解压到了别处 | 确认在 `~` 下执行 `tar xzf` |